# TD10a

In [5]:
import copy
import time 
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

if device != "cuda":
    exit("This MRE is designed to show GPU memory differences, so please run on a CUDA-capable GPU to see the results.")

class DenseNet(nn.Module):
    def __init__(self, d, mult):
        super().__init__()
        self.fc1 = nn.Linear(d, mult*d, bias=False)
        self.fc2 = nn.Linear(mult*d, mult*d, bias=False)
        self.fc3 = nn.Linear(mult*d, d, bias=False)
        self.act = nn.ReLU()

    def _block(self, x):
        return self.fc3(self.act(self.fc2(self.act(self.fc1(x)))))

    def forward(self, x, use_ckpt=False):
        if use_ckpt:
            return checkpoint(self._block, x, use_reentrant=False)
        return self._block(x)


device: cuda


In [6]:
def bench(use_ckpt: bool):
    # Do some copies to keep everything identical across runs
    model = copy.deepcopy(base_model).to(device)
    opt = torch.optim.Adam(model.parameters())
    x = base_x.clone().to(device).requires_grad_(True)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    out = model(x, use_ckpt=use_ckpt)
    loss = out.pow(2).mean()
    torch.cuda.synchronize()
    t_fwd = time.perf_counter() - t0

    mem_after_fwd = torch.cuda.memory_allocated()
    peak_fwd = torch.cuda.max_memory_allocated()
    torch.cuda.reset_peak_memory_stats()

    t0 = time.perf_counter()
    loss.backward()
    torch.cuda.synchronize()
    t_bwd = time.perf_counter() - t0
    peak_bwd_extra = torch.cuda.max_memory_allocated()

    opt.step()

    final_params = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    return dict(
        t_fwd=t_fwd, 
        t_bwd=t_bwd,
        peak_fwd=peak_fwd,
        mem_after_fwd=mem_after_fwd, 
        peak_bwd_extra=peak_bwd_extra,
        final_params=final_params
    )

In [7]:
base_model = DenseNet(d=100, mult=50)
base_x = torch.randn(100000, 100)

res_remember = bench(use_ckpt=False)
res_no_remember = bench(use_ckpt=True)

print("=== TIME (seconds) ===")
print(f"forward:  remember={res_remember['t_fwd']:.4f} no_remember={res_no_remember['t_fwd']:.4f}")
print(f"backward: remember={res_remember['t_bwd']:.4f} no_remember={res_no_remember['t_bwd']:.4f}")

print("\n=== MEMORY (CUDA bytes) ===")
print(f"allocated after forward:\n remember={res_remember['mem_after_fwd']:,} no_remember={res_no_remember['mem_after_fwd']:,}")
print(f"peak during forward:\n remember={res_remember['peak_fwd']:,} no_remember={res_no_remember['peak_fwd']:,}")
print(f"peak during backward\n remember={res_remember['peak_bwd_extra']:,} no_remember={res_no_remember['peak_bwd_extra']:,}")

=== TIME (seconds) ===
forward:  remember=0.5603 no_remember=0.3016
backward: remember=0.4824 no_remember=0.7934

=== MEMORY (CUDA bytes) ===
allocated after forward:
 remember=4,203,069,952 no_remember=201,703,936
peak during forward:
 remember=6,163,752,448 no_remember=4,163,069,440
peak during backward
 remember=8,206,436,864 no_remember=8,206,436,864


In [8]:
# Exact parameter update equality check
all_equal = True
for k in res_remember["final_params"]:
    if not torch.equal(res_remember["final_params"][k], res_no_remember["final_params"][k]):
        all_equal = False
        print("MISMATCH in:", k)
        break

print("=== PARAM UPDATE EXACT MATCH? ===")
print(all_equal)

=== PARAM UPDATE EXACT MATCH? ===
True
